# Applying data minimization to a trained ML model

This notebook can be used to run the original Minimization logic, prooving that it still works after updated dependencies and compatability fixes.  


This will be demonstarted using the Adult dataset (original dataset can be found here: https://archive.ics.uci.edu/ml/datasets/adult).   
We will subsample the dataset to make it run faster.

We will use a Random Forest Classifier as the base model and show how we can greatly increase the generalization of data while barely sacrificing accuracy!

## Step 1: Load Data

In [ ]:
from ucimlrepo import fetch_ucirepo

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from apt.minimization import GeneralizeToRepresentative

from apt.utils.metrics import calculate_disclosure_risk

In [2]:
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets["income"]

# Convert labels to integers
y = y.apply(lambda x: 1 if x.startswith("<=50K") else 0).astype(int)

# Only keep 10% of dataset for faster execution
X, _, y, _ = train_test_split(X, y, train_size=0.1, random_state=42, stratify=y)

In [3]:
# Identify numerical and categorical values

categorical_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

feature_names = numerical_cols + categorical_cols

In [4]:
# Impute the null categorical values
X[categorical_cols] = X[categorical_cols].fillna("NULL")

In [5]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=42
)

## Training Base Model

In [6]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [7]:
encoded_train = preprocessor.fit_transform(X_train)
encoded_test = preprocessor.transform(X_test)

In [8]:
model = RandomForestClassifier(random_state=42)
model.fit(encoded_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [9]:
print("Base model accuracy: ", model.score(encoded_test, y_test))

Base model accuracy:  0.8259979529170931


## Step 3: Run Generalization

In [11]:
train_predicted_y = model.predict(encoded_train)

minimizer = GeneralizeToRepresentative(
    model,
    categorical_features=categorical_cols,
    encoder=preprocessor,
    target_accuracy=0.75,
)


minimizer.fit(X_train, train_predicted_y, features_names=X_train.columns.tolist())

Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.799104
Improving generalizations
Pruned tree to level: 1, new relative accuracy: 0.796545
Pruned tree to level: 2, new relative accuracy: 0.796545
Pruned tree to level: 3, new relative accuracy: 0.797185
Pruned tree to level: 4, new relative accuracy: 0.797185
Pruned tree to level: 5, new relative accuracy: 0.797825
Pruned tree to level: 6, new relative accuracy: 0.795266
Pruned tree to level: 7, new relative accuracy: 0.789507
Pruned tree to level: 8, new relative accuracy: 0.788868
Pruned tree to level: 9, new relative accuracy: 0.784389
Pruned tree to level: 10, new relative accuracy: 0.776711
Pruned tree to level: 11, new relative accuracy: 0.776072
Pruned tree to level: 12, new relative accuracy: 0.775432
Pruned tree to level: 13, new relative accuracy: 0.767115
Pruned tree to level: 14, new relative accuracy: 0.775432
Pruned tree to

,estimator,<apt.utils.mo...x7fac2decfa90>
,target_accuracy,0.75
,cells,"[{'categories': {'education': ['1st-4th', 'Assoc-voc', ...], 'marital-status': ['Divorced', 'Married-AF-spouse', ...], 'native-country': ['Ireland', 'Nicaragua', ...], 'occupation': ['Tech-support', 'Priv-house-serv', ...], ...}, 'hist': array([[1., 4.]]), 'id': 9, 'label': [np.int64(1)], ...}, {'categories': {'education': ['1st-4th', 'Assoc-voc', ...], 'marital-status': ['Divorced', 'Widowed', ...], 'native-country': ['Ireland', 'Nicaragua', ...], 'occupation': ['Exec-managerial'], ...}, 'hist': array([[1., 2.]]), 'id': 18, 'label': [np.int64(1)], ...}, ...]"
,categorical_features,"['workclass', 'education', ...]"
,encoder,ColumnTransfo...e-country'])])
,features_to_minimize,"['age', 'workclass', ...]"
,feature_slices,None
,train_only_features_to_minimize,True
,is_regression,False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'


In [15]:
minimizer.generalizations

{'ranges': {'age': [np.float64(28.5),
   np.float64(30.5),
   np.float64(32.5),
   np.float64(41.5),
   np.float64(42.0),
   np.float64(44.5),
   np.float64(49.5),
   np.float64(51.0),
   np.float64(58.5),
   np.float64(59.0)],
  'fnlwgt': [np.float64(25243.5),
   np.float64(56149.0),
   np.float64(74158.5),
   np.float64(78516.0),
   np.float64(107098.0),
   np.float64(126294.0),
   np.float64(164485.5),
   np.float64(180159.5),
   np.float64(197399.5),
   np.float64(203081.5),
   np.float64(233638.0),
   np.float64(233973.5),
   np.float64(259570.0)],
  'education-num': [np.float64(8.5),
   np.float64(10.5),
   np.float64(11.5),
   np.float64(12.5),
   np.float64(13.5),
   np.float64(15.5)],
  'capital-gain': [np.float64(3120.0),
   np.float64(3761.5),
   np.float64(4718.5),
   np.float64(5095.5),
   np.float64(7055.5)],
  'capital-loss': [np.float64(742.5),
   np.float64(1129.0),
   np.float64(1813.5),
   np.float64(1846.0),
   np.float64(2014.0),
   np.float64(2391.5),
   np.float6

## Step 4: Comparing Accuracy & Disclosure Risk

The "disclosure risk" defined in the original paper calculates on average - how unique each record is and hence, how easy is it to disclose an individual. It is rather similar to k-similarity or anonimity-set in the purpose.

The lower it is (0), the more abstract and similar each datapoints are to each other.  
If it's high (1), then each datapoint is completely unique!

In [19]:
baseline_acc = model.score(encoded_test, y_test)
baseline_disclosure_risk = calculate_disclosure_risk(X_test)


print("Baseline model accuracy: ", baseline_acc)
print("Basline model disclosure risk: ", baseline_disclosure_risk)

Baseline model accuracy:  0.8259979529170931
Basline model disclosure risk:  1.0


In [20]:
generalized_X_test = minimizer.transform(X_test)
encoded_generalized_x_test = preprocessor.transform(generalized_X_test)

generalized_acc = model.score(encoded_generalized_x_test, y_test)
generalized_disclosure_risk = calculate_disclosure_risk(generalized_X_test)

print("Generalized model accuracy: ", generalized_acc)
print("Generalized model disclosure risk: ", generalized_disclosure_risk)

Generalized model accuracy:  0.7983623336745138
Generalized model disclosure risk:  0.0511770726714432
